In [ ]:
在设计 Prompt 时，我们还可以通过明确指导语言模型进行自主思考，来获得更好的效果。 
举个例子，假设我们要语言模型判断一个数学问题的解答是否正确。仅仅提供问题和解答是不够的，语 言模型可能会匆忙做出错误判断。

相反，我们可以在 Prompt 中先要求语言模型自己尝试解决这个问题，思考出自己的解法，然后再与提 供的解答进行对比，判断正确性。
这种先让语言模型自主思考的方式，能帮助它更深入理解问题，做出 更准确的判断。

In [4]:
import os

from dotenv import load_dotenv, find_dotenv

from zhipuai import ZhipuAI

_ = load_dotenv(find_dotenv())

client = ZhipuAI(
    api_key=os.environ["ZHIPUAI_API_KEY"]
)

def gen_glm_params(prompt):
    '''

    请求参数：
        prompt: 对应的用户提示词
    '''
    messages = [{"role": "user", "content": prompt}]
    return messages


def get_completion(prompt, model="glm-4-flash", temperature=0.95):
    '''
    获取 GLM 模型调用结果

    请求参数：
        prompt: 对应的提示词
        model: 调用的模型，默认为 glm-4，也可以按需选择 glm-3-turbo 等其他模型
        temperature: 模型输出的温度系数，控制输出的随机程度，取值范围是 0.0-1.0。温度系数越低，输出内容越一致。
    '''

    messages = gen_glm_params(prompt)
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    if len(response.choices) > 0:
        return response.choices[0].message.content
    return "generate answer error"

In [2]:
prompt = f"""
判断学生的解决方案是否正确。
问题:
我正在建造一个太阳能发电站，需要帮助计算财务。
土地费用为 100美元/平方英尺
我可以以 250美元/平方英尺的价格购买太阳能电池板
我已经谈判好了维护合同，每年需要支付固定的10万美元，并额外支付每平方英尺10美元
作为平方英尺数的函数，首年运营的总费用是多少。
学生的解决方案：
设x为发电站的大小，单位为平方英尺。
费用：
土地费用：100x
太阳能电池板费用：250x
维护费用：100,000美元+100x
总费用：100x+250x+100,000美元+100x=450x+100,000美元
"""

response = get_completion(prompt)
print(response)

学生的解决方案是正确的。

我们来逐步验证一下：

1. 土地费用：根据题目，土地费用为每平方英尺100美元，因此土地费用是 \( 100x \) 美元。

2. 太阳能电池板费用：每平方英尺250美元，因此太阳能电池板费用是 \( 250x \) 美元。

3. 维护费用：每年固定10万美元，另外每平方英尺10美元。因此，维护费用是 \( 100,000 \) 美元加上 \( 10x \) 美元。

将这些费用加在一起，得到总费用的表达式：

总费用 = 土地费用 + 太阳能电池板费用 + 维护费用
总费用 = \( 100x + 250x + 100,000 + 10x \)

将相同的项合并，我们得到：

总费用 = \( 450x + 100,000 \)

所以学生的解决方案 \( 100x + 250x + 100,000 + 100x = 450x + 100,000 \) 是正确的。


In [ ]:
但是注意，学生的解决方案实际上是错误的。
（维护费用项100x应为10x，总费用450x应为360x）。我们可以通过指导模型先自行找出一个解法来解决这个问题。

在接下来这个 Prompt 中，我们要求模型先自行解决这个问题，再根据自己的解法与学生的解法进行对比，从而判断学生的解法是否正确。
同时，我们给定了输出的格式要求。通过拆分任务、明确步骤，让 模型有更多时间思考，有时可以获得更准确的结果。

In [6]:
prompt = f"""
请判断学生的解决方案是否正确，请通过如下步骤解决这个问题：
步骤：
首先，自己解决问题。
然后将您的解决方案与学生的解决方案进行比较，对比计算得到的总费用与学生计算的总费用是否一致，
并评估学生的解决方案是否正确。
在自己完成问题之前，请勿决定学生的解决方案是否正确。
使用以下格式：
问题：问题文本
学生的解决方案：学生的解决方案文本
实际解决方案和步骤：实际解决方案和步骤文本
学生计算的总费用：学生计算得到的总费用
实际计算的总费用：实际计算出的总费用
学生计算的费用和实际计算的费用是否相同：是或否
学生的解决方案和实际解决方案是否相同：是或否
学生的成绩：正确或不正确
问题：
我正在建造一个太阳能发电站，需要帮助计算财务。
- 土地费用为每平方英尺100美元
- 我可以以每平方英尺250美元的价格购买太阳能电池板
- 我已经谈判好了维护合同，每年需要支付固定的10万美元，并额外支付每平方英尺10美元;
作为平方英尺数的函数，首年运营的总费用是多少。
学生的解决方案：
设x为发电站的大小，单位为平方英尺。
费用：
1. 土地费用：100x美元
2. 太阳能电池板费用：250x美元
3. 维护费用：100,000+100x=10万美元+10x美元
总费用：100x美元+250x美元+10万美元+100x美元=450x+10万美元
实际解决方案和步骤：
"""

response = get_completion(prompt)
print(response)

问题：
我正在建造一个太阳能发电站，需要帮助计算财务。
- 土地费用为每平方英尺100美元
- 我可以以每平方英尺250美元的价格购买太阳能电池板
- 我已经谈判好了维护合同，每年需要支付固定的10万美元，并额外支付每平方英尺10美元;
作为平方英尺数的函数，首年运营的总费用是多少。

学生的解决方案：
设x为发电站的大小，单位为平方英尺。
费用：
1. 土地费用：100x美元
2. 太阳能电池板费用：250x美元
3. 维护费用：100,000+100x=10万美元+10x美元
总费用：100x美元+250x美元+10万美元+100x美元=450x+10万美元

实际解决方案和步骤：
首先，我们需要计算每个部分的费用，然后将它们相加得到总费用。

1. 土地费用：由于土地费用是每平方英尺100美元，所以如果发电站的大小是x平方英尺，土地费用将是100x美元。
2. 太阳能电池板费用：太阳能电池板费用是每平方英尺250美元，因此对于x平方英尺的发电站，太阳能电池板费用是250x美元。
3. 维护费用：维护合同固定费用是10万美元，另外每平方英尺需要支付10美元，所以维护费用是100,000美元加上10x美元。

将这些费用相加，我们得到总费用的计算公式：
总费用 = 土地费用 + 太阳能电池板费用 + 维护费用
总费用 = 100x + 250x + 100,000 + 10x
总费用 = 360x + 100,000

学生计算的总费用：450x + 10万美元
实际计算的总费用：360x + 10万美元
学生计算的费用和实际计算的费用是否相同：否
学生的解决方案和实际解决方案是否相同：否
学生的成绩：不正确

学生的解决方案中，维护费用的计算有误，应该是100,000美元加上每平方英尺10美元，而不是100x美元。正确的总费用计算应该是360x + 100,000美元。
